# Phase 5 — EWMA Volatility Scaling of Historical VaR

**Advanced Equity Portfolio Risk Management Platform**  
**QFI Internship — Team 2 (Market Risk)** · *Submitted by: Atharv Pratap Singh Chauhan*

---

## Objective

Phase 4 established **unscaled Historical Simulation (HS)** as the empirical benchmark. Phase 5 retains its non-parametric tail shape but makes the scenarios responsive to the volatility regime using the **Hull–White volatility-scaling approach**.

| Deliverable | Specification |
|---|---|
| Volatility model | EWMA variance with daily RiskMetrics decay $\lambda=0.94$ |
| Scaling | Standardise each return by its contemporaneous forecast volatility, then rescale to today's volatility |
| Rolling windows | 250 / 500 / 750 trading days |
| Confidence levels | 95% / 99% / 99.5% |
| Required charts | EWMA volatility; raw vs scaled returns; unscaled vs scaled VaR; exception comparison |
| Phase-8 hand-off | `outputs/EWMA_Scaled_VaR.csv` plus `outputs/EWMA_Scaled_VaR_Data_Dictionary.csv` |

All VaR estimates are **one-day-ahead forecasts**: the day-$t$ forecast uses information available only through day $t-1$.


## 1. Financial intuition — why scale Historical Simulation?

Plain HS assigns equal importance to every return inside its lookback window. This preserves realised skewness and fat tails, but ignores **volatility clustering**: quiet and turbulent observations are pooled as though they came from the same regime.

That creates two familiar weaknesses:

1. **Slow response at the start of stress.** A few new losses may not move an empirical percentile dominated by hundreds of calm observations, so exceptions can cluster.
2. **The Ghost Effect.** A crisis observation receives full weight until it reaches the edge of the window, then disappears completely. VaR may remain artificially elevated through a recovery and drop abruptly on day $W+1$, even if nothing new happened in the market.

EWMA lets volatility react smoothly:

$$\sigma_t^2=\lambda\sigma_{t-1}^2+(1-\lambda)r_{t-1}^2.$$

With $\lambda=0.94$, yesterday's squared return receives 6% weight and the shock-weight half-life is approximately 11.2 trading days. No observation falls out on a single arbitrary date; its influence decays geometrically.

The Hull–White transformation separates **shock magnitude** from **volatility regime**:

$$z_i=\frac{r_i}{\sigma_i}, \qquad r_{i\rightarrow t}^{\mathrm{scaled}}=z_i\sigma_t=r_i\frac{\sigma_t}{\sigma_i}.$$

Thus, every historical shock is replayed under today's volatility. The empirical tail shape is retained, while its scale expands promptly in stress and contracts smoothly in calm markets:

$$\mathrm{VaR}^{\mathrm{scaled}}_{t,c}=-\sigma_t Q_{1-c}\left(z_{t-W},\ldots,z_{t-1}\right).$$

This improves responsiveness without imposing Normality. It does **not** eliminate model risk: $\lambda$ remains a judgement, the method is still limited by observed standardised shocks, and sudden volatility spikes can make capital requirements more variable.


## 2. Environment, house style, and central configuration

The configuration object is the single source of truth. Colours, labels, output paths, confidence levels, and timing conventions are standardised to match Phase 4.


In [ ]:
from __future__ import annotations

import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

PALETTE = {
    "ink": "#1B2A41", "muted": "#8C99A6", "profit": "#2E7D68",
    "loss": "#B03A2E", "hs": "#E08A2E", "ewma": "#6A2C91",
    "vol": "#2C5F8A", "w250": "#E08A2E", "w500": "#C0392B",
    "w750": "#5B2C6F", "light": "#E9EDF2"
}
WINDOW_COLORS = {250: PALETTE["w250"], 500: PALETTE["w500"], 750: PALETTE["w750"]}

sns.set_theme(style="whitegrid", context="talk")
mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": PALETTE["muted"], "axes.labelcolor": PALETTE["ink"],
    "axes.titlecolor": PALETTE["ink"], "axes.titleweight": "bold",
    "axes.titlesize": 14, "axes.labelsize": 11, "axes.titlepad": 12,
    "grid.color": "#E3E7EB", "grid.linewidth": 0.8,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.frameon": True, "legend.framealpha": 0.94, "legend.fontsize": 9,
    "font.family": "DejaVu Sans"
})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

@dataclass(frozen=True)
class RiskConfig:
    lam: float = 0.94
    windows: tuple[int, ...] = (250, 500, 750)
    confidences: tuple[float, ...] = (0.95, 0.99, 0.995)
    headline_window: int = 250
    headline_confidence: float = 0.99
    trading_days: int = 252
    seed_days: int = 60
    burn_in: int = 60
    currency: str = "₹"
    data_file: str = "Portfolio_Construction.csv"
    phase4_file: str = "Rolling_Historical_VaR.csv"
    output_dir: Path = Path("outputs")
    chart_dir: Path = Path("outputs/charts")

    @property
    def half_life(self): return float(np.log(0.5) / np.log(self.lam))

CFG = RiskConfig()
CFG.output_dir.mkdir(parents=True, exist_ok=True)
CFG.chart_dir.mkdir(parents=True, exist_ok=True)

def pct(x, dp=2): return "n/a" if not np.isfinite(x) else f"{100*x:.{dp}f}%"
def money(x, dp=0): return "n/a" if not np.isfinite(x) else f"{CFG.currency}{x:,.{dp}f}"
def cl_tag(c): return f"{100*c:g}".replace(".", "p")
def save_fig(fig, name):
    path = CFG.chart_dir / f"{name}.png"
    fig.savefig(path, facecolor="white", bbox_inches="tight", dpi=200)
    print(f"Saved chart -> {path}")

print(CFG)
print(f"λ={CFG.lam:.2f} | half-life={CFG.half_life:.1f} trading days | newest squared-return weight={1-CFG.lam:.1%}")


## 3. Load and validate the Phase-3 portfolio

The loader searches common project folders, normalises likely column names, and reconstructs missing `Return`, `PnL`, or `Value` fields where possible. If the Phase-3 CSV is absent, a clearly labelled synthetic series is used solely so the notebook remains executable; replace it with the real file before submission.


In [ ]:
def locate(name: str):
    candidates = [Path(name), Path("data")/name, Path("outputs")/name,
                  Path("..")/name, Path("..")/"data"/name, Path("..")/"outputs"/name]
    return next((p for p in candidates if p.exists()), None)

def normalise(s): return "".join(ch for ch in str(s).lower() if ch.isalnum())

ALIASES = {
    "Return": {"return", "returns", "portfolioreturn", "dailyreturn", "ret"},
    "PnL": {"pnl", "pandl", "profitloss", "portfoliopnl", "dailypnl"},
    "Value": {"value", "portfoliovalue", "nav", "marketvalue", "portfolio"}
}

def synthetic_portfolio(n=1800, seed=42):
    rng = np.random.default_rng(seed)
    eps = rng.standard_t(5, n) / np.sqrt(5/3)
    vol = np.empty(n); vol[0] = 0.010
    for t in range(1, n):
        vol[t] = np.sqrt(2e-6 + 0.08*(vol[t-1]*eps[t-1])**2 + 0.90*vol[t-1]**2)
    vol[int(.43*n):int(.49*n)] *= 2.4
    ret = 0.0003 + vol*eps
    idx = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=n)
    value = 10_000_000*np.cumprod(1+ret)
    out = pd.DataFrame({"Return": ret, "Value": value}, index=idx)
    out["PnL"] = out["Value"].diff().fillna(0)
    out.index.name = "Date"
    return out

def load_portfolio():
    path = locate(CFG.data_file)
    demo = path is None
    if demo:
        print(f"[DEMO] {CFG.data_file} not found; using synthetic volatility-clustered data.")
        return synthetic_portfolio(), True
    raw = pd.read_csv(path)
    date_col = next((c for c in raw.columns if normalise(c) in {"date","datetime","timestamp","index","unnamed0"}), raw.columns[0])
    raw[date_col] = pd.to_datetime(raw[date_col], errors="coerce")
    raw = raw.dropna(subset=[date_col]).set_index(date_col).sort_index()
    mapping = {}
    for target, aliases in ALIASES.items():
        src = next((c for c in raw.columns if normalise(c) in aliases), None)
        if src is not None: mapping[target] = src
    df = pd.DataFrame(index=raw.index)
    for target, src in mapping.items():
        cleaned = raw[src].astype(str).str.replace(r"[,₹$£€\s()]", "", regex=True)
        df[target] = pd.to_numeric(cleaned, errors="coerce")
    if "Return" not in df and "Value" in df: df["Return"] = df["Value"].pct_change()
    if "PnL" not in df and "Value" in df: df["PnL"] = df["Value"].diff()
    if "Value" not in df and "Return" in df:
        df["Value"] = 10_000_000*(1+df["Return"].fillna(0)).cumprod()
    if "PnL" not in df and {"Return","Value"} <= set(df): df["PnL"] = df["Return"]*df["Value"].shift(1)
    required = {"Return", "PnL", "Value"}
    if not required <= set(df): raise ValueError(f"Cannot construct {required-set(df)} from columns {list(raw.columns)}")
    df = df.replace([np.inf,-np.inf], np.nan).dropna(subset=list(required))
    df = df[~df.index.duplicated(keep="last")].sort_index()
    print(f"Loaded {len(df):,} clean observations from {path} | mapping={mapping}")
    return df, False

portfolio, DEMO_MODE = load_portfolio()
returns = portfolio["Return"].astype(float)
pnl = portfolio["PnL"].astype(float)
value = portfolio["Value"].astype(float)
capital = value.shift(1).bfill()

print(f"Sample: {portfolio.index.min():%d-%b-%Y} to {portfolio.index.max():%d-%b-%Y}")
print(f"Latest NAV: {money(value.iloc[-1])} | Annualised realised volatility: {pct(returns.std()*np.sqrt(CFG.trading_days))}")
display(portfolio.tail())


## 4. EWMA volatility engine

The recursion below is explicit to preserve timing. `sigma[t]` uses `return[t-1]`, so it is known before day $t$ begins. The initial variance is seeded from the first 60 observations; the burn-in region is excluded from diagnostics.


In [ ]:
def ewma_volatility(r: pd.Series, lam=CFG.lam, seed_days=CFG.seed_days):
    x = r.to_numpy(float)
    var = np.empty(len(x))
    var[0] = np.var(x[:min(seed_days, len(x))], ddof=1)
    for t in range(1, len(x)):
        var[t] = lam*var[t-1] + (1-lam)*x[t-1]**2
    return pd.Series(np.sqrt(var), index=r.index, name="EWMA_Vol_Daily")

sigma = ewma_volatility(returns)
sigma_ann = sigma*np.sqrt(CFG.trading_days)
rolling_vol = returns.rolling(250, min_periods=60).std().shift(1)*np.sqrt(CFG.trading_days)
valid = pd.Series(False, index=returns.index); valid.iloc[CFG.burn_in:] = True
z = (returns/sigma).rename("Standardized_Return")

vol_table = pd.DataFrame({
    "Latest": [pct(sigma_ann.iloc[-1]), pct(rolling_vol.iloc[-1])],
    "Mean": [pct(sigma_ann[valid].mean()), pct(rolling_vol.dropna().mean())],
    "Maximum": [pct(sigma_ann[valid].max()), pct(rolling_vol.dropna().max())]
}, index=["EWMA λ=0.94", "Equal-weighted 250d"])
display(vol_table)
print(f"Standardised return variance={z[valid].var():.3f} (unit-variance target = 1.000)")


### Chart 1 — EWMA volatility and regime responsiveness

The lower panel plots EWMA volatility relative to the slowly moving 250-day estimate. Values above 1 flag periods when plain HS is likely under-responsive; values below 1 highlight the post-stress region in which old crisis observations can create a ghost effect.


In [ ]:
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True, gridspec_kw={"height_ratios":[2.4,1]})
ax.plot(sigma_ann, color=PALETTE["ewma"], lw=2.0, label=f"EWMA volatility (λ={CFG.lam})")
ax.plot(rolling_vol, color=PALETTE["muted"], lw=1.7, ls="--", label="Equal-weighted 250-day volatility")
ax.fill_between(sigma_ann.index, 0, sigma_ann.values, color=PALETTE["ewma"], alpha=.08)
ax.scatter([sigma_ann.index[-1]], [sigma_ann.iloc[-1]], s=75, color=PALETTE["ewma"], zorder=5)
ax.annotate(f"Latest: {pct(sigma_ann.iloc[-1])}", (sigma_ann.index[-1], sigma_ann.iloc[-1]),
            xytext=(-8,15), textcoords="offset points", ha="right", fontsize=10, fontweight="bold")
ax.set_title("Chart 1 — EWMA volatility reacts quickly; equal-weighted volatility carries old regimes", loc="left")
ax.set_ylabel("Annualised volatility"); ax.yaxis.set_major_formatter(mticker.PercentFormatter(1)); ax.legend(loc="upper left")
ratio = sigma_ann/rolling_vol
ax2.plot(ratio, color=PALETTE["vol"], lw=1.3); ax2.axhline(1, color=PALETTE["ink"], lw=1)
ax2.fill_between(ratio.index, 1, ratio.values, where=ratio.ge(1), color=PALETTE["loss"], alpha=.25, label="EWMA > 250d: HS slow to arm")
ax2.fill_between(ratio.index, 1, ratio.values, where=ratio.lt(1), color=PALETTE["profit"], alpha=.25, label="EWMA < 250d: HS ghost region")
ax2.set_ylabel("EWMA / 250d"); ax2.legend(loc="upper left", ncol=2)
fig.tight_layout(); save_fig(fig, "01_ewma_volatility"); plt.show()


## 5. Raw returns versus Hull–White scaled returns

For the latest reporting date, the historical shock series is replayed at current volatility. Standardised returns should exhibit less volatility clustering than raw returns; the comparison preserves empirical asymmetry and tail events while changing their scale.


In [ ]:
scaled_latest = z*sigma.iloc[-1]

def acf_sq(s, lags=20):
    x = np.asarray(s.dropna(), float); x = x-x.mean(); den = x@x
    return np.array([(x[k:]@x[:-k])/den for k in range(1,lags+1)])

fig = plt.figure(figsize=(17, 9))
gs = fig.add_gridspec(2, 2, hspace=.32, wspace=.22)
a0=fig.add_subplot(gs[0,:]); a1=fig.add_subplot(gs[1,0]); a2=fig.add_subplot(gs[1,1])
a0.scatter(returns.index, returns*100, s=8, alpha=.43, color=PALETTE["muted"], label="Raw historical returns")
a0.scatter(scaled_latest.index, scaled_latest*100, s=8, alpha=.48, color=PALETTE["ewma"], label=f"Replayed at latest σ={pct(sigma.iloc[-1])}/day")
a0.axhline(0,color=PALETTE["ink"],lw=.8); a0.set_ylabel("Daily return (%)")
a0.set_title("Chart 2 — Raw history versus history replayed in today's volatility regime", loc="left"); a0.legend(ncol=2)
lo = min(returns[valid].quantile(.002), scaled_latest[valid].quantile(.002))*100
hi = max(returns[valid].quantile(.998), scaled_latest[valid].quantile(.998))*100
bins=np.linspace(lo,hi,65)
a1.hist(returns[valid]*100,bins=bins,density=True,alpha=.55,color=PALETTE["muted"],label="Raw")
a1.hist(scaled_latest[valid]*100,bins=bins,density=True,alpha=.48,color=PALETTE["ewma"],label="Scaled")
a1.set_title("Distribution comparison",loc="left"); a1.set_xlabel("Return (%)"); a1.set_ylabel("Density"); a1.legend()
lags=np.arange(1,21); ci=1.96/np.sqrt(valid.sum())
a2.bar(lags-.2,acf_sq(returns[valid]**2),.4,color=PALETTE["hs"],label="Raw squared returns")
a2.bar(lags+.2,acf_sq(z[valid]**2),.4,color=PALETTE["ewma"],label="Standardised squared shocks")
a2.axhspan(-ci,ci,color=PALETTE["light"],label="95% white-noise band")
a2.set_title("Volatility clustering diagnostic",loc="left"); a2.set_xlabel("Lag (days)"); a2.set_ylabel("Autocorrelation"); a2.legend(fontsize=8)
fig.tight_layout(); save_fig(fig,"02_raw_vs_scaled_returns"); plt.show()


## 6. Rolling unscaled and EWMA-scaled Historical VaR

For each day $t$, both models use exactly the same prior-data window. This isolates the effect of volatility scaling and prevents look-ahead bias.

- **Unscaled HS:** $-Q_{1-c}(r_{t-W},\ldots,r_{t-1})$
- **EWMA-scaled HS:** $-\sigma_t Q_{1-c}(z_{t-W},\ldots,z_{t-1})$

Return VaR is converted to currency using start-of-day portfolio value. Positive numbers represent loss limits.


In [ ]:
result = pd.DataFrame(index=returns.index)
result.index.name = "Date"
result["Portfolio_Return"] = returns
result["Portfolio_PnL"] = pnl
result["Start_of_Day_Value"] = capital
result["EWMA_Vol_Daily"] = sigma
result["EWMA_Vol_Annualized"] = sigma_ann
result["Standardized_Return"] = z

for w in CFG.windows:
    prior_r = returns.shift(1).rolling(w, min_periods=w)
    prior_z = z.shift(1).rolling(w, min_periods=w)
    for c in CFG.confidences:
        a=1-c; tag=cl_tag(c)
        hs_ret = (-prior_r.quantile(a)).clip(lower=0)
        ew_ret = (-sigma*prior_z.quantile(a)).clip(lower=0)
        result[f"HS_VaR_{w}d_{tag}_Return"] = hs_ret
        result[f"EWMA_VaR_{w}d_{tag}_Return"] = ew_ret
        result[f"HS_VaR_{w}d_{tag}"] = hs_ret*capital
        result[f"EWMA_VaR_{w}d_{tag}"] = ew_ret*capital
        result[f"HS_Breach_{w}d_{tag}"] = (pnl < -result[f"HS_VaR_{w}d_{tag}"]).where(hs_ret.notna())
        result[f"EWMA_Breach_{w}d_{tag}"] = (pnl < -result[f"EWMA_VaR_{w}d_{tag}"]).where(ew_ret.notna())

w=CFG.headline_window; tag=cl_tag(CFG.headline_confidence)
hs_col=f"HS_VaR_{w}d_{tag}"; ew_col=f"EWMA_VaR_{w}d_{tag}"
print(f"Latest {CFG.headline_confidence:.1%} {w}d HS VaR: {money(result[hs_col].iloc[-1])}")
print(f"Latest {CFG.headline_confidence:.1%} {w}d EWMA-scaled VaR: {money(result[ew_col].iloc[-1])}")


### Chart 3 — Unscaled HS VaR versus EWMA-scaled VaR

The upper panel compares realised loss with both forecast limits. The lower panel makes responsiveness explicit: a scaled-to-unscaled ratio above 1 shows EWMA arming faster in stress; below 1 shows it releasing the stale crisis premium more smoothly.


In [ ]:
loss=-pnl
common=result[[hs_col,ew_col]].dropna().index
fig,(ax,ax2)=plt.subplots(2,1,figsize=(16,9),sharex=True,gridspec_kw={"height_ratios":[2.5,1]})
ax.plot(common,loss.loc[common],color=PALETTE["muted"],lw=.8,alpha=.65,label="Realised daily loss")
ax.plot(common,result.loc[common,hs_col],color=PALETTE["hs"],lw=1.7,label="Unscaled HS VaR")
ax.plot(common,result.loc[common,ew_col],color=PALETTE["ewma"],lw=2.0,label="EWMA-scaled HS VaR")
ax.axhline(0,color=PALETTE["ink"],lw=.7); ax.set_ylabel(f"Loss / VaR ({CFG.currency})")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter(CFG.currency+"{x:,.0f}"))
ax.set_title(f"Chart 3 — {CFG.headline_confidence:.1%} VaR comparison ({w}-day window)",loc="left"); ax.legend(ncol=3)
vr=result[ew_col]/result[hs_col]
ax2.plot(vr,color=PALETTE["vol"],lw=1.4); ax2.axhline(1,color=PALETTE["ink"],lw=1)
ax2.fill_between(vr.index,1,vr.values,where=vr.ge(1),color=PALETTE["loss"],alpha=.25,label="Scaled VaR higher: faster stress response")
ax2.fill_between(vr.index,1,vr.values,where=vr.lt(1),color=PALETTE["profit"],alpha=.25,label="Scaled VaR lower: stale crisis premium released")
ax2.set_ylabel("Scaled / unscaled"); ax2.legend(ncol=2)
fig.tight_layout(); save_fig(fig,"03_var_comparison"); plt.show()


## 7. Exception comparison

An exception occurs when realised P&L is more negative than the forecast VaR. The chart distinguishes exceptions unique to each model from days on which both fail. A good volatility-scaled model should reduce concentrated bursts of exceptions during regime changes, although no model is expected to eliminate exceptions entirely.


In [ ]:
def exception_stats(var_col, breach_col, confidence):
    v=result[var_col].notna(); b=result.loc[v,breach_col].astype(bool); n=int(v.sum()); x=int(b.sum())
    rate=x/n if n else np.nan; expected=(1-confidence)*n
    if n and 0<x<n:
        p=1-confidence
        lr=-2*((n-x)*np.log((1-p)/(1-x/n))+x*np.log(p/(x/n)))
        kupiec_p=1-stats.chi2.cdf(lr,1)
    elif n and x==0:
        lr=-2*n*np.log(1-(1-confidence)); kupiec_p=1-stats.chi2.cdf(lr,1)
    else: kupiec_p=np.nan
    return n,x,rate,expected,kupiec_p

rows=[]
for w0 in CFG.windows:
    for c in CFG.confidences:
        t=cl_tag(c)
        for model,prefix in [("Unscaled HS","HS"),("EWMA-scaled HS","EWMA")]:
            n,x,rate,exp,kp=exception_stats(f"{prefix}_VaR_{w0}d_{t}",f"{prefix}_Breach_{w0}d_{t}",c)
            rows.append({"Window":w0,"Confidence":c,"Model":model,"Forecast days":n,
                         "Exceptions":x,"Expected":exp,"Exception rate":rate,"Kupiec p-value":kp})
exception_table=pd.DataFrame(rows)
display(exception_table.style.format({"Confidence":"{:.1%}","Expected":"{:.1f}","Exception rate":"{:.2%}","Kupiec p-value":"{:.3f}"}))


In [ ]:
hsb=result.loc[common,f"HS_Breach_{w}d_{tag}"].fillna(False).astype(bool)
ewb=result.loc[common,f"EWMA_Breach_{w}d_{tag}"].fillna(False).astype(bool)
both=hsb & ewb; hs_only=hsb & ~ewb; ew_only=ewb & ~hsb
fig,(ax,axb)=plt.subplots(2,1,figsize=(16,8),gridspec_kw={"height_ratios":[2.7,1]})
ax.plot(common,loss.loc[common],color=PALETTE["muted"],lw=.8,alpha=.55,label="Realised loss")
ax.plot(common,result.loc[common,hs_col],color=PALETTE["hs"],lw=1.25,alpha=.9,label="Unscaled HS VaR")
ax.plot(common,result.loc[common,ew_col],color=PALETTE["ewma"],lw=1.55,alpha=.9,label="EWMA-scaled VaR")
ax.scatter(common[hs_only],loss.loc[common[hs_only]],s=55,color=PALETTE["hs"],marker="o",edgecolor="white",label=f"HS only ({hs_only.sum()})",zorder=5)
ax.scatter(common[ew_only],loss.loc[common[ew_only]],s=65,color=PALETTE["ewma"],marker="X",edgecolor="white",label=f"EWMA only ({ew_only.sum()})",zorder=6)
ax.scatter(common[both],loss.loc[common[both]],s=75,color=PALETTE["loss"],marker="D",edgecolor="white",label=f"Both ({both.sum()})",zorder=7)
ax.set_title(f"Chart 4 — Exception comparison ({CFG.headline_confidence:.1%}, {w}-day window)",loc="left")
ax.set_ylabel(f"Loss / VaR ({CFG.currency})"); ax.yaxis.set_major_formatter(mticker.StrMethodFormatter(CFG.currency+"{x:,.0f}")); ax.legend(ncol=3)
# 60-day rolling exception frequency highlights clustering
hs_roll=hsb.astype(int).rolling(60,min_periods=1).sum(); ew_roll=ewb.astype(int).rolling(60,min_periods=1).sum()
axb.step(common,hs_roll,where="mid",color=PALETTE["hs"],lw=1.6,label="HS exceptions in trailing 60 days")
axb.step(common,ew_roll,where="mid",color=PALETTE["ewma"],lw=1.8,label="EWMA exceptions in trailing 60 days")
axb.set_ylabel("60d count"); axb.set_xlabel(""); axb.legend(ncol=2)
fig.tight_layout(); save_fig(fig,"04_exception_comparison"); plt.show()


## 8. Phase-8 export and data dictionary

The export is deliberately wide: every date retains realised portfolio fields, EWMA state variables, unscaled and scaled VaR forecasts, and model-specific breach flags. This allows Phase 8 to run coverage and independence tests without recomputing Phase 5.


In [ ]:
export_path=CFG.output_dir/"EWMA_Scaled_VaR.csv"
result.to_csv(export_path, date_format="%Y-%m-%d")

dict_rows=[
    ("Date","date","Trading date; row index."),
    ("Portfolio_Return","decimal","Realised daily portfolio return."),
    ("Portfolio_PnL",f"{CFG.currency}","Realised daily portfolio profit/loss."),
    ("Start_of_Day_Value",f"{CFG.currency}","Portfolio value used to convert return VaR to currency."),
    ("EWMA_Vol_Daily","decimal/day","One-day-ahead EWMA volatility using information through t-1."),
    ("EWMA_Vol_Annualized","decimal/year","Daily EWMA volatility multiplied by sqrt(252)."),
    ("Standardized_Return","sigma units","Return divided by contemporaneous one-day-ahead EWMA volatility."),
    ("HS_VaR_<W>d_<CL>_Return","positive decimal","Unscaled Historical Simulation return VaR forecast."),
    ("EWMA_VaR_<W>d_<CL>_Return","positive decimal","Hull–White EWMA-scaled return VaR forecast."),
    ("HS_VaR_<W>d_<CL>",f"{CFG.currency}","Unscaled HS VaR in currency; positive loss limit."),
    ("EWMA_VaR_<W>d_<CL>",f"{CFG.currency}","EWMA-scaled HS VaR in currency; positive loss limit."),
    ("HS_Breach_<W>d_<CL>","boolean","True when realised P&L is below negative unscaled VaR."),
    ("EWMA_Breach_<W>d_<CL>","boolean","True when realised P&L is below negative scaled VaR.")]
data_dictionary=pd.DataFrame(dict_rows,columns=["Column / Pattern","Unit / Type","Definition"])
dict_path=CFG.output_dir/"EWMA_Scaled_VaR_Data_Dictionary.csv"
data_dictionary.to_csv(dict_path,index=False)
print(f"Exported {result.shape[0]:,} rows x {result.shape[1]:,} columns -> {export_path}")
print(f"Exported data dictionary -> {dict_path}")
display(data_dictionary)


## 9. Dynamic Executive Summary

The following cell is intentionally data-driven. Its wording and metrics update automatically whenever the portfolio, window, confidence level, or EWMA decay factor changes.


In [ ]:
hs_n,hs_x,hs_rate,hs_exp,hs_kp=exception_stats(hs_col,f"HS_Breach_{w}d_{tag}",CFG.headline_confidence)
ew_n,ew_x,ew_rate,ew_exp,ew_kp=exception_stats(ew_col,f"EWMA_Breach_{w}d_{tag}",CFG.headline_confidence)
latest_hs=float(result[hs_col].iloc[-1]); latest_ew=float(result[ew_col].iloc[-1])
delta=(latest_ew/latest_hs-1) if latest_hs else np.nan
regime=sigma.iloc[-1]/sigma[valid].mean()
regime_word="elevated" if regime>1.15 else "calm" if regime<.85 else "normal"
better="fewer" if ew_x<hs_x else "more" if ew_x>hs_x else "the same number of"
source="synthetic demonstration data" if DEMO_MODE else "the Phase-3 portfolio"

summary=f"""
### Executive Summary — Phase 5

- **Model implemented.** A one-day-ahead EWMA filter with $\\lambda={CFG.lam:.2f}$ (half-life **{CFG.half_life:.1f} trading days**) was combined with empirical Hull–White scaling over **{', '.join(map(str,CFG.windows))}-day** windows at **{', '.join(f'{c:.1%}' for c in CFG.confidences)}** confidence.
- **Current regime.** Latest annualised EWMA volatility is **{pct(sigma_ann.iloc[-1])}**, or **{regime:.2f}×** its sample average; the portfolio is therefore in a **{regime_word}** volatility regime.
- **Headline VaR.** At {CFG.headline_confidence:.1%} confidence using the {w}-day window, current unscaled HS VaR is **{money(latest_hs)}**, versus **{money(latest_ew)}** after EWMA scaling (**{delta:+.1%}**).
- **Exceptions.** Across {min(hs_n,ew_n):,} comparable forecast days, unscaled HS recorded **{hs_x}** exceptions ({hs_rate:.2%}); EWMA-scaled HS recorded **{ew_x}** ({ew_rate:.2%}) — **{better} exceptions** after scaling. The nominal exception rate is **{1-CFG.headline_confidence:.2%}**.
- **Interpretation.** Scaling does not guarantee a lower VaR or fewer breaches on every sample. Its benefit is *timing*: it raises risk promptly when volatility clusters and releases the stale crisis premium smoothly, reducing the plateau-and-cliff mechanics of the Ghost Effect.
- **Phase-8 readiness.** `EWMA_Scaled_VaR.csv` contains realised P&L, both forecast families, and breach flags; the companion data dictionary defines every field. Results shown here use **{source}**.

**Risk judgement:** use EWMA-scaled HS as the more regime-responsive daily monitoring measure, while retaining unscaled HS as a transparent benchmark. Phase 8 should formally test unconditional coverage and exception independence across every window/confidence pair before model approval.
"""
display(Markdown(summary))


## 10. Limitations and hand-off

1. **Historical support remains finite.** Scaling cannot invent a standardised shock more severe than the observed sample.
2. **EWMA has one persistence parameter.** A single $\lambda$ may not describe all regimes; sensitivity analysis or GARCH-family alternatives can be considered later.
3. **VaR is not tail severity.** Phase 8 should supplement exception testing with Expected Shortfall and tail-loss analysis.
4. **Portfolio composition is assumed represented by the return series.** Structural changes in weights can invalidate older observations.
5. **Backtest power is limited at 99.5%.** Very few expected exceptions mean that long samples and careful interpretation are essential.

**Next step:** Phase 8 can consume the exported file directly to perform Kupiec unconditional-coverage, Christoffersen independence/conditional-coverage, and Basel traffic-light diagnostics consistently across models.
